# CodeWeave — Master Notebook

One notebook, one Colab runtime, everything in memory — no cross-notebook
Drive mounting required to move between sections. Sections:

1. Setup
2. Repository Dataset Builder (clone, scan, Git history)
3. Code AST + Symbol/Relationship Extraction (Tree-sitter)
4. Semantic Code Similarity (CodeBERT + FAISS)
5. Bug/Risk Prediction (baseline → LR → RF → XGBoost + SHAP)

**Run top to bottom in one session.** Drive is mounted once at the end of
each section purely as a *backup* — everything needed to keep working stays
in memory (`files_df`, `symbols_df`, `commits_df`, etc.), so you never have
to reload from disk mid-session. If your runtime disconnects, re-run from
the top (repo cloning is skip-if-exists, so a re-run is fast the second time).


In [1]:
!pip -q install GitPython pandas tqdm python-dotenv tree-sitter tree-sitter-python \
    tree-sitter-javascript tree-sitter-typescript transformers faiss-cpu radon xgboost shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.9/221.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.1/668.1 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.0/345.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 4.3 MB/s eta 0:00:00


In [2]:
import json
import re
import pickle
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from git import Repo

# Local disk (not Drive) for speed during the run — cloning + parsing thousands
# of files is much faster on local disk than over the Drive FUSE mount.
BASE_DIR = Path("/content/codeweave_dataset")
REPO_DIR = BASE_DIR / "repositories"
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
METRICS_DIR = BASE_DIR / "metrics"
for d in [REPO_DIR, DATA_DIR, MODELS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Working directory:", BASE_DIR)

Working directory: /content/codeweave_dataset


In [4]:
MOUNT_DRIVE_BACKUP = True  # set False to skip Drive entirely

if MOUNT_DRIVE_BACKUP:
    from google.colab import drive
    import os
    if not os.path.isdir('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_BACKUP_DIR = Path('/content/drive/MyDrive/codeweave_dataset')
    DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    print('Drive backup target:', DRIVE_BACKUP_DIR)
else:
    DRIVE_BACKUP_DIR = None
    print('Drive backup disabled.')


def backup_to_drive():
    """Copy current data/models/metrics to Drive. Call this after any section
    you don't want to risk losing. Safe no-op if MOUNT_DRIVE_BACKUP is False."""
    if DRIVE_BACKUP_DIR is None:
        return
    import shutil
    for sub in ["data", "models", "metrics"]:
        src = BASE_DIR / sub
        dst = DRIVE_BACKUP_DIR / sub
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Backed up to {DRIVE_BACKUP_DIR}")

Mounted at /content/drive
Drive backup target: /content/drive/MyDrive/codeweave_dataset


## 2. Repository Dataset Builder

Clone repositories, scan files, extract Git commit + change history.


In [5]:
REPOSITORIES = [
    {"name": "repository_1", "url": "https://github.com/pallets/flask.git"},
    {"name": "repository_2", "url": "https://github.com/fastapi/fastapi.git"},
    {"name": "repository_3", "url": "https://github.com/encode/django-rest-framework.git"},
]

In [6]:
def clone_repository(name, url):
    repo_path = REPO_DIR / name
    if repo_path.exists():
        print(f"{name}: already cloned, skipping.")
        return repo_path
    print(f"Cloning {name} from {url} ...")
    Repo.clone_from(url, repo_path)
    return repo_path


for repo_info in REPOSITORIES:
    clone_repository(repo_info["name"], repo_info["url"])

Cloning repository_1 from https://github.com/pallets/flask.git ...
Cloning repository_2 from https://github.com/fastapi/fastapi.git ...
Cloning repository_3 from https://github.com/encode/django-rest-framework.git ...


In [7]:
IGNORED_DIRECTORIES = {".git", "node_modules", "venv", ".venv", "__pycache__",
                        "dist", "build", ".next", "coverage", ".pytest_cache", ".mypy_cache"}
IGNORED_EXTENSIONS = {".pyc", ".pyo", ".so", ".dll", ".exe", ".bin", ".zip",
                       ".tar", ".gz", ".jpg", ".jpeg", ".png", ".gif", ".mp4", ".mp3"}
EXTENSION_TO_LANGUAGE = {
    ".py": "Python", ".js": "JavaScript", ".jsx": "JavaScript",
    ".ts": "TypeScript", ".tsx": "TypeScript", ".md": "Markdown",
    ".json": "JSON", ".yml": "YAML", ".yaml": "YAML",
}


def should_ignore(path: Path) -> bool:
    parts = set(path.parts)
    if parts.intersection(IGNORED_DIRECTORIES):
        return True
    if path.suffix.lower() in IGNORED_EXTENSIONS:
        return True
    return False


def is_test_file(path_str: str) -> bool:
    name = Path(path_str).name.lower()
    return (name.startswith("test_") or name.endswith("_test.py")
            or ".test." in name or ".spec." in name
            or "/tests/" in path_str.lower() or "\\tests\\" in path_str.lower())


def get_module(path_str: str) -> str:
    parts = Path(path_str).parts
    return parts[0] if len(parts) > 1 else "root"


def scan_repository(repo_name, repo_path):
    records = []
    for path in repo_path.rglob("*"):
        if not path.is_file() or should_ignore(path.relative_to(repo_path)):
            continue
        language = EXTENSION_TO_LANGUAGE.get(path.suffix.lower())
        if language is None:
            continue
        try:
            content = path.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        rel_path = str(path.relative_to(repo_path))
        records.append({
            "repository_name": repo_name,
            "path": rel_path,
            "language": language,
            "extension": path.suffix.lower(),
            "size_bytes": path.stat().st_size,
            "loc": len(content.splitlines()),
            "is_test": is_test_file(rel_path),
            "module": get_module(rel_path),
        })
    return records


all_file_records = []
for repo_info in REPOSITORIES:
    repo_path = REPO_DIR / repo_info["name"]
    all_file_records.extend(scan_repository(repo_info["name"], repo_path))

files_df = pd.DataFrame(all_file_records)
print(f"Scanned {len(files_df)} files.")
files_df.to_csv(DATA_DIR / "files.csv", index=False)
files_df.head(3)

Scanned 3227 files.


,repository_name,path,language,extension,size_bytes,loc,is_test,module
0,repository_1,.readthedocs.yaml,YAML,.yaml,242,10,False,root
1,repository_1,.pre-commit-config.yaml,YAML,.yaml,844,24,False,root
2,repository_1,README.md,Markdown,.md,1639,53,False,root


In [8]:
def extract_commits(repo_name, repo_path):
    repo = Repo(repo_path)
    records = []
    for commit in repo.iter_commits():
        records.append({
            "repository_name": repo_name,
            "sha": commit.hexsha,
            "author": commit.author.name,
            "author_email": commit.author.email,
            "timestamp": datetime.fromtimestamp(commit.committed_date),
            "message": commit.message.strip().split("\n")[0],
        })
    return records


all_commit_records = []
for repo_info in tqdm(REPOSITORIES):
    repo_path = REPO_DIR / repo_info["name"]
    all_commit_records.extend(extract_commits(repo_info["name"], repo_path))

commits_df = pd.DataFrame(all_commit_records)
print(f"Extracted {len(commits_df)} commits.")
commits_df.to_csv(DATA_DIR / "commits.csv", index=False)
commits_df.head(3)

  0%|          | 0/3 [00:00<?, ?it/s]

Extracted 22294 commits.


,repository_name,sha,author,author_email,timestamp,message
0,repository_1,d318b683471101618febed18996405ad26462110,David Lord,davidism@gmail.com,2026-08-16 18:35:31,explain seek
1,repository_1,2a8a38b051fc248865730bf3511bf2e2ea325e81,David Lord,davidism@gmail.com,2026-08-11 22:32:49,support query in methodview
2,repository_1,d8eaaba824655046958d1a97f11780de460c3271,David Lord,davidism@gmail.com,2026-08-11 20:19:14,add `app.query` route decorator (#6133)


In [9]:
def extract_changes(repo_name, repo_path):
    repo = Repo(repo_path)
    records = []
    for commit in repo.iter_commits():
        if not commit.parents:
            continue
        parent = commit.parents[0]
        try:
            diffs = parent.diff(commit, create_patch=True)
        except Exception:
            continue
        for d in diffs:
            path = d.b_path or d.a_path
            if path is None or d.diff is None:
                continue
            # Count actual added/removed LINES (not raw '+'/'-' byte occurrences),
            # excluding the +++ / --- file-header lines.
            added, removed = 0, 0
            for line in d.diff.split(b"\n"):
                if line.startswith(b"+++") or line.startswith(b"---"):
                    continue
                if line.startswith(b"+"):
                    added += 1
                elif line.startswith(b"-"):
                    removed += 1
            records.append({
                "repository_name": repo_name,
                "commit_sha": commit.hexsha,
                "file_path": path,
                "lines_added": added,
                "lines_removed": removed,
            })
    return records


all_change_records = []
for repo_info in tqdm(REPOSITORIES):
    repo_path = REPO_DIR / repo_info["name"]
    all_change_records.extend(extract_changes(repo_info["name"], repo_path))

changes_df = pd.DataFrame(all_change_records)
print(f"Extracted {len(changes_df)} change events.")
changes_df.to_csv(DATA_DIR / "changes.csv", index=False)
changes_df.head(3)

  0%|          | 0/3 [00:00<?, ?it/s]

Extracted 79697 change events.


,repository_name,commit_sha,file_path,lines_added,lines_removed
0,repository_1,d318b683471101618febed18996405ad26462110,src/flask/helpers.py,2,2
1,repository_1,2a8a38b051fc248865730bf3511bf2e2ea325e81,src/flask/views.py,1,1
2,repository_1,d8eaaba824655046958d1a97f11780de460c3271,CHANGES.rst,1,0


In [10]:
backup_to_drive()

Backed up to /content/drive/MyDrive/codeweave_dataset


## 3. Code AST + Symbol / Relationship Extraction

Uses `files_df` already in memory — no reload needed.


In [11]:
import tree_sitter_python as tspython
import tree_sitter_javascript as tsjavascript
import tree_sitter_typescript as tstypescript
from tree_sitter import Language, Parser

PY_LANGUAGE = Language(tspython.language())
JS_LANGUAGE = Language(tsjavascript.language())
TS_LANGUAGE = Language(tstypescript.language_typescript())

LANGUAGE_MAP = {".py": PY_LANGUAGE, ".js": JS_LANGUAGE, ".jsx": JS_LANGUAGE,
                 ".ts": TS_LANGUAGE, ".tsx": TS_LANGUAGE}

FUNCTION_NODE_TYPES = {"function_definition", "function_declaration", "method_definition"}
CLASS_NODE_TYPES = {"class_definition", "class_declaration"}
IMPORT_NODE_TYPES = {"import_statement", "import_from_statement"}
CALL_NODE_TYPES = {"call", "call_expression"}

print("Parsers ready for:", list(LANGUAGE_MAP.keys()))

Parsers ready for: ['.py', '.js', '.jsx', '.ts', '.tsx']


In [12]:
def get_parser(extension):
    lang = LANGUAGE_MAP.get(extension)
    return Parser(lang) if lang is not None else None


def node_text(node, source):
    return source[node.start_byte:node.end_byte].decode("utf8", errors="ignore")


def find_name(node, source):
    name_node = node.child_by_field_name("name")
    return node_text(name_node, source) if name_node is not None else None


def extract_symbols_and_relationships(repo_name, relative_path, source_bytes, extension):
    parser = get_parser(extension)
    if parser is None:
        return [], []
    try:
        tree = parser.parse(source_bytes)
    except Exception:
        return [], []

    root = tree.root_node
    symbols, relationships = [], []

    def walk(node, enclosing_class=None):
        node_type = node.type

        if node_type in CLASS_NODE_TYPES:
            name = find_name(node, source_bytes)
            if name:
                symbols.append({
                    "repository_name": repo_name, "file_path": relative_path,
                    "symbol_type": "class", "name": name,
                    "start_line": node.start_point[0] + 1, "end_line": node.end_point[0] + 1,
                    "signature": name, "parent_class": None,
                })
            for child in node.children:
                walk(child, enclosing_class=name)
            return

        if node_type in FUNCTION_NODE_TYPES:
            name = find_name(node, source_bytes)
            if name:
                params_node = node.child_by_field_name("parameters")
                params_text = node_text(params_node, source_bytes) if params_node else "()"
                symbols.append({
                    "repository_name": repo_name, "file_path": relative_path,
                    "symbol_type": "method" if enclosing_class else "function", "name": name,
                    "start_line": node.start_point[0] + 1, "end_line": node.end_point[0] + 1,
                    "signature": f"{name}{params_text}", "parent_class": enclosing_class,
                })

        if node_type in IMPORT_NODE_TYPES:
            relationships.append({
                "repository_name": repo_name, "source_file": relative_path,
                "relation_type": "imports",
                "target": node_text(node, source_bytes).strip().replace("\n", " "),
                "line": node.start_point[0] + 1,
            })

        if node_type in CALL_NODE_TYPES:
            fn_node = node.child_by_field_name("function")
            if fn_node is not None:
                relationships.append({
                    "repository_name": repo_name, "source_file": relative_path,
                    "relation_type": "calls", "target": node_text(fn_node, source_bytes),
                    "line": node.start_point[0] + 1,
                })

        for child in node.children:
            walk(child, enclosing_class=enclosing_class)

    walk(root)
    return symbols, relationships

In [13]:
PARSEABLE_EXTENSIONS = set(LANGUAGE_MAP.keys())
parseable_files_df = files_df[files_df["extension"].isin(PARSEABLE_EXTENSIONS)].reset_index(drop=True)
print(f"{len(parseable_files_df)} of {len(files_df)} files are parseable (Python/JS/TS).")

all_symbol_records, all_relationship_records, skipped_files = [], [], []

for row in tqdm(parseable_files_df.itertuples(), total=len(parseable_files_df)):
    full_path = REPO_DIR / row.repository_name / row.path
    try:
        source_bytes = full_path.read_bytes()
    except Exception as e:
        skipped_files.append({"repository_name": row.repository_name, "path": row.path, "error": str(e)})
        continue
    symbols, relationships = extract_symbols_and_relationships(
        row.repository_name, row.path, source_bytes, row.extension
    )
    all_symbol_records.extend(symbols)
    all_relationship_records.extend(relationships)

symbols_df = pd.DataFrame(all_symbol_records)
symbols_df["symbol_id"] = symbols_df.index.map(lambda i: f"sym_{i:06d}")
relationships_df = pd.DataFrame(all_relationship_records)

print("Total symbols:", len(symbols_df), "| Total relationships:", len(relationships_df))
print("Files skipped:", len(skipped_files))

symbols_df.to_csv(DATA_DIR / "symbols.csv", index=False)
relationships_df.to_csv(DATA_DIR / "relationships.csv", index=False)
backup_to_drive()

1387 of 3227 files are parseable (Python/JS/TS).


  0%|          | 0/1387 [00:00<?, ?it/s]

Total symbols: 12027 | Total relationships: 40532
Files skipped: 0
Backed up to /content/drive/MyDrive/codeweave_dataset


## 4. Semantic Code Similarity — CodeBERT + FAISS

Uses `symbols_df` already in memory. Requires internet access to download
`microsoft/codebert-base` from Hugging Face on first run (works fine in Colab).


In [14]:
def get_snippet(row):
    file_path = REPO_DIR / row["repository_name"] / row["file_path"]
    try:
        lines = file_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    except Exception:
        return None
    start = max(row["start_line"] - 1, 0)
    end = min(row["end_line"], len(lines))
    return "\n".join(lines[start:end]) if start < end else None


symbols_df["snippet"] = symbols_df.apply(get_snippet, axis=1)
before = len(symbols_df)
symbols_df = symbols_df.dropna(subset=["snippet"]).reset_index(drop=True)
symbols_df = symbols_df[symbols_df["snippet"].str.strip() != ""].reset_index(drop=True)
symbols_df["symbol_id"] = symbols_df.index.map(lambda i: f"sym_{i:06d}")
print(f"{len(symbols_df)} of {before} symbols have a readable snippet.")

12027 of 12027 symbols have a readable snippet.


In [15]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "microsoft/codebert-base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded:", MODEL_NAME)

Using device: cuda


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded: microsoft/codebert-base


In [16]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_batch(texts, max_length=256):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
    outputs = model(**inputs)
    pooled = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
    return torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy()


BATCH_SIZE = 32
snippets = symbols_df["snippet"].tolist()
all_embeddings = []
for i in tqdm(range(0, len(snippets), BATCH_SIZE)):
    all_embeddings.append(embed_batch(snippets[i:i + BATCH_SIZE]))

embeddings = np.vstack(all_embeddings).astype("float32")
print("Embeddings shape:", embeddings.shape)
np.save(DATA_DIR / "embeddings.npy", embeddings)

  0%|          | 0/376 [00:00<?, ?it/s]

Embeddings shape: (12027, 768)


In [17]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
faiss.write_index(index, str(DATA_DIR / "symbol_index.faiss"))
print("Indexed vectors:", index.ntotal)

symbol_id_to_row = {sid: i for i, sid in enumerate(symbols_df["symbol_id"])}


def find_similar(symbol_id, k=5):
    row_idx = symbol_id_to_row[symbol_id]
    distances, indices = index.search(embeddings[row_idx:row_idx + 1], k + 1)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == row_idx:
            continue
        row = symbols_df.iloc[idx]
        results.append({"symbol_id": row["symbol_id"], "name": row["name"],
                         "repository_name": row["repository_name"], "file_path": row["file_path"],
                         "similarity": float(dist)})
        if len(results) == k:
            break
    return results


example_id = symbols_df["symbol_id"].iloc[0]
print(f"Query: {symbols_df.iloc[0]['name']} ({example_id})")
for r in find_similar(example_id, k=5):
    print(f"  {r['similarity']:.3f}  {r['name']}  ({r['repository_name']}/{r['file_path']})")

Indexed vectors: 12027
Query: github_link (sym_000000)
  0.994  test_url_processors  (repository_1/tests/test_basic.py)
  0.994  HostNameVersioning  (repository_3/rest_framework/versioning.py)
  0.993  get_client  (repository_2/tests/test_tutorial/test_sql_databases/test_tutorial001.py)
  0.993  get_client  (repository_2/tests/test_tutorial/test_sql_databases/test_tutorial002.py)
  0.993  add_permalinks_page  (repository_2/scripts/docs.py)


### Benchmark: CodeBERT vs. TF-IDF baseline

Auto-bootstraps candidate pairs from same-named symbols across files as a
starting benchmark — **hand-verify `benchmark_pairs.csv` before trusting these
numbers**, same-name doesn't guarantee true semantic similarity.


In [18]:
benchmark_path = DATA_DIR / "benchmark_pairs.csv"

if benchmark_path.exists():
    benchmark_df = pd.read_csv(benchmark_path)
else:
    name_groups = symbols_df.groupby("name").filter(lambda g: len(g) >= 2)
    pairs = []
    for name, group in name_groups.groupby("name"):
        rows = group.sample(min(2, len(group)), random_state=0)
        if len(rows) == 2:
            pairs.append({"query_symbol_id": rows.iloc[0]["symbol_id"], "true_symbol_id": rows.iloc[1]["symbol_id"]})
        if len(pairs) >= 30:
            break
    benchmark_df = pd.DataFrame(pairs)
    benchmark_df.to_csv(benchmark_path, index=False)

print(f"Benchmark pairs: {len(benchmark_df)}")


def recall_at_k(query_ids, true_ids, k):
    hits = 0
    for q, t in zip(query_ids, true_ids):
        if q not in symbol_id_to_row or t not in symbol_id_to_row:
            continue
        retrieved = [r["symbol_id"] for r in find_similar(q, k=k)]
        hits += int(t in retrieved)
    return hits / max(len(query_ids), 1)


def mrr(query_ids, true_ids, k):
    rr = []
    for q, t in zip(query_ids, true_ids):
        if q not in symbol_id_to_row or t not in symbol_id_to_row:
            continue
        retrieved = [r["symbol_id"] for r in find_similar(q, k=k)]
        rr.append(1.0 / (retrieved.index(t) + 1) if t in retrieved else 0.0)
    return float(np.mean(rr)) if rr else 0.0


K = 10
codebert_recall = recall_at_k(benchmark_df["query_symbol_id"], benchmark_df["true_symbol_id"], K)
codebert_mrr = mrr(benchmark_df["query_symbol_id"], benchmark_df["true_symbol_id"], K)
print(f"CodeBERT  Recall@{K}: {codebert_recall:.3f}   MRR: {codebert_mrr:.3f}")

Benchmark pairs: 30
CodeBERT  Recall@10: 0.433   MRR: 0.329


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(symbols_df["snippet"])


def tfidf_find_similar(symbol_id, k=5):
    row_idx = symbol_id_to_row[symbol_id]
    sims = cosine_similarity(tfidf_matrix[row_idx], tfidf_matrix).flatten()
    results = []
    for idx in np.argsort(-sims):
        if idx == row_idx:
            continue
        results.append(symbols_df.iloc[idx]["symbol_id"])
        if len(results) == k:
            break
    return results


def tfidf_recall_at_k(query_ids, true_ids, k):
    hits = 0
    for q, t in zip(query_ids, true_ids):
        if q not in symbol_id_to_row or t not in symbol_id_to_row:
            continue
        hits += int(t in tfidf_find_similar(q, k=k))
    return hits / max(len(query_ids), 1)


def tfidf_mrr(query_ids, true_ids, k):
    rr = []
    for q, t in zip(query_ids, true_ids):
        if q not in symbol_id_to_row or t not in symbol_id_to_row:
            continue
        retrieved = tfidf_find_similar(q, k=k)
        rr.append(1.0 / (retrieved.index(t) + 1) if t in retrieved else 0.0)
    return float(np.mean(rr)) if rr else 0.0


tfidf_recall = tfidf_recall_at_k(benchmark_df["query_symbol_id"], benchmark_df["true_symbol_id"], K)
tfidf_mrr_score = tfidf_mrr(benchmark_df["query_symbol_id"], benchmark_df["true_symbol_id"], K)
print(f"TF-IDF    Recall@{K}: {tfidf_recall:.3f}   MRR: {tfidf_mrr_score:.3f}")

retrieval_metrics = {
    "k": K, "benchmark_size": len(benchmark_df),
    "codebert": {"recall_at_k": codebert_recall, "mrr": codebert_mrr},
    "tfidf_baseline": {"recall_at_k": tfidf_recall, "mrr": tfidf_mrr_score},
    "model_version": MODEL_NAME, "generated_at": datetime.now().isoformat(),
}
with open(METRICS_DIR / "retrieval.json", "w") as f:
    json.dump(retrieval_metrics, f, indent=4)

backup_to_drive()

TF-IDF    Recall@10: 0.567   MRR: 0.362
Backed up to /content/drive/MyDrive/codeweave_dataset


## 5. Bug / Risk Prediction

Uses `files_df`, `commits_df`, `changes_df` already in memory.


In [20]:
changes_full_df = changes_df.merge(
    commits_df[["repository_name", "sha", "timestamp", "message", "author"]],
    left_on=["repository_name", "commit_sha"], right_on=["repository_name", "sha"], how="left",
)
print(changes_full_df.shape)

(79697, 9)


In [21]:
from radon.complexity import cc_visit

JS_COMPLEXITY_PATTERN = re.compile(r"\b(if|for|while|case|catch)\b|(\&\&)|(\|\|)")


def compute_complexity(repo_name, file_path, language):
    full_path = REPO_DIR / repo_name / file_path
    try:
        source = full_path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return None, "unreadable"
    if language == "Python":
        try:
            blocks = cc_visit(source)
            return (float(np.mean([b.complexity for b in blocks])) if blocks else 1.0), "radon"
        except Exception:
            return None, "radon_parse_error"
    matches = JS_COMPLEXITY_PATTERN.findall(source)
    return float(len(matches) + 1), "keyword_proxy"


complexity_results = files_df.apply(
    lambda row: compute_complexity(row["repository_name"], row["path"], row["language"]), axis=1
)
files_df["complexity"] = complexity_results.apply(lambda x: x[0])
files_df["complexity_method"] = complexity_results.apply(lambda x: x[1])
files_df["complexity"] = files_df["complexity"].fillna(files_df["complexity"].median())
print(files_df["complexity_method"].value_counts())

complexity_method
keyword_proxy    1852
radon            1375
Name: count, dtype: int64


In [22]:
min_ts = changes_full_df["timestamp"].min()
max_ts = changes_full_df["timestamp"].max()
cutoff = min_ts + timedelta(days=int((max_ts - min_ts).days * 0.8))

train_changes = changes_full_df[changes_full_df["timestamp"] < cutoff].copy()
eval_changes = changes_full_df[changes_full_df["timestamp"] >= cutoff].copy()

print(f"Cutoff: {cutoff.date()} | pre-cutoff: {len(train_changes)} | post-cutoff: {len(eval_changes)}")
assert train_changes["timestamp"].max() < cutoff
assert eval_changes["timestamp"].min() >= cutoff
print("Temporal split verified.")

Cutoff: 2023-05-16 | pre-cutoff: 55189 | post-cutoff: 24508
Temporal split verified.


In [23]:
def build_features(files_df, changes_subset, as_of):
    feats = files_df[["repository_name", "path", "loc", "module", "complexity"]].copy()
    feats = feats.set_index(["repository_name", "path"])
    grouped = changes_subset.groupby(["repository_name", "file_path"])
    churn = grouped.apply(lambda g: (g["lines_added"].fillna(0) + g["lines_removed"].fillna(0)).sum())
    change_freq = grouped.size()
    contributors = grouped["author"].nunique()
    last_change = grouped["timestamp"].max()

    feats["churn"] = churn.reindex(feats.index).fillna(0)
    feats["change_frequency"] = change_freq.reindex(feats.index).fillna(0)
    feats["contributor_count"] = contributors.reindex(feats.index).fillna(0)
    feats["days_since_last_change"] = (as_of - last_change.reindex(feats.index)).dt.days.fillna(9999)
    return feats.reset_index()


train_features = build_features(files_df, train_changes, cutoff)

BUGFIX_PATTERN = r"\bfix(?:e[sd])?\b|\bbug(?:fix)?\b|\bpatch\b|\bhotfix\b"
bugfix_eval = eval_changes[eval_changes["message"].fillna("").str.contains(BUGFIX_PATTERN, case=False, regex=True)]
buggy_keys = set(zip(bugfix_eval["repository_name"], bugfix_eval["file_path"]))
train_features["label"] = train_features.apply(lambda r: int((r["repository_name"], r["path"]) in buggy_keys), axis=1)

print(f"Positive label rate: {train_features['label'].mean():.3f}")

Positive label rate: 0.251


/tmp/ipykernel_3179/641963719.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  churn = grouped.apply(lambda g: (g["lines_added"].fillna(0) + g["lines_removed"].fillna(0)).sum())


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, auc, f1_score
from sklearn.calibration import calibration_curve
import xgboost as xgb

FEATURE_COLS = ["loc", "complexity", "churn", "change_frequency", "contributor_count", "days_since_last_change"]
X = train_features[FEATURE_COLS].values
y = train_features["label"].values

stratify = y if (y.sum() >= 2 and (len(y) - y.sum()) >= 2) else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=stratify)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (2258, 6), Test: (969, 6)


In [25]:
def evaluate(name, y_true, y_scores):
    if len(set(y_true)) < 2:
        print(f"{name}: skipped (single class in test set)")
        return None
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    pr_auc = auc(recall, precision)
    f1 = f1_score(y_true, (y_scores >= 0.5).astype(int), zero_division=0)
    k = max(int(y_true.sum()), 1)
    top_k_idx = np.argsort(-y_scores)[:k]
    precision_at_k = y_true[top_k_idx].sum() / k
    print(f"{name}: PR-AUC={pr_auc:.3f}  F1={f1:.3f}  Precision@{k}={precision_at_k:.3f}")
    return {"pr_auc": pr_auc, "f1": f1, "precision_at_k": precision_at_k, "k": k}


risk_results = {}
cf_idx = FEATURE_COLS.index("change_frequency")
baseline_scores = X_test[:, cf_idx] / (X_test[:, cf_idx].max() + 1e-9)
risk_results["baseline_change_frequency"] = evaluate("Baseline", y_test, baseline_scores)

lr = LogisticRegression(max_iter=1000, class_weight="balanced")
lr.fit(X_train_scaled, y_train)
risk_results["logistic_regression"] = evaluate("Logistic Regression", y_test, lr.predict_proba(X_test_scaled)[:, 1])

rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=0)
rf.fit(X_train, y_train)
risk_results["random_forest"] = evaluate("Random Forest", y_test, rf.predict_proba(X_test)[:, 1])

scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                               scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=0)
xgb_model.fit(X_train, y_train)
xgb_scores = xgb_model.predict_proba(X_test)[:, 1]
risk_results["xgboost"] = evaluate("XGBoost", y_test, xgb_scores)

Baseline: PR-AUC=0.443  F1=0.008  Precision@244=0.447
Logistic Regression: PR-AUC=0.535  F1=0.530  Precision@244=0.512
Random Forest: PR-AUC=0.529  F1=0.505  Precision@244=0.512
XGBoost: PR-AUC=0.650  F1=0.531  Precision@244=0.541


In [26]:
import shap

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

top_risk_idx = int(np.argmax(xgb_scores))
contributions = sorted(zip(FEATURE_COLS, shap_values[top_risk_idx]), key=lambda x: abs(x[1]), reverse=True)
print(f"Highest-risk test file: score={xgb_scores[top_risk_idx]:.3f}")
for feature, value in contributions[:3]:
    print(f"  {feature}: {'increases' if value > 0 else 'decreases'} risk (SHAP={value:+.3f})")

Highest-risk test file: score=0.997
  complexity: increases risk (SHAP=+3.434)
  loc: increases risk (SHAP=+1.211)
  days_since_last_change: increases risk (SHAP=+0.961)


In [27]:
with open(MODELS_DIR / "risk_model.pkl", "wb") as f:
    pickle.dump({"model": xgb_model, "scaler": scaler, "feature_cols": FEATURE_COLS}, f)

train_features.to_csv(DATA_DIR / "risk_features.csv", index=False)
with open(METRICS_DIR / "risk.json", "w") as f:
    json.dump({"cutoff_date": cutoff.isoformat(), "results": risk_results,
                "generated_at": datetime.now().isoformat()}, f, indent=4)

backup_to_drive()
print("Done. All sections complete.")

Backed up to /content/drive/MyDrive/codeweave_dataset
Done. All sections complete.


## 6. Change Impact Prediction

Given a file that changed, predict which other files are likely affected.
Combines three signals, per the implementation plan §7.3:

1. **Dependency graph** — resolved import relationships (`relationships_df`)
2. **Historical co-change** — files that have changed together in the same
   commit before (`changes_full_df`, computed only from the pre-cutoff period)
3. **Semantic similarity** — file-level embeddings aggregated from the
   symbol embeddings already built in Section 4

Uses `files_df`, `relationships_df`, `symbols_df`, `embeddings`, and
`changes_full_df` already in memory — no reload needed.

**Simplification to flag:** the dependency graph here is treated as
*undirected* (if A imports B, that's one edge either direction) for
simplicity. In reality "who imports me" and "what do I import" carry
different impact implications — worth revisiting in the ablation study
(Section 7) if graph features turn out to matter a lot.


### 6.1 Resolve imports into a file-level dependency graph

`relationships_df` records import statements as raw text (e.g.
`"from .param_functions import Cookie"`) — not yet linked to an actual file.
This resolves each import to a real file path within the same repo when
possible (handles Python relative imports with `.`/`..`, absolute
`package.module` imports, and JS/TS relative `'./path'` imports). External
packages and stdlib imports correctly won't resolve — only in-repo
dependencies matter for impact prediction.


In [28]:
import re
from pathlib import PurePosixPath
from collections import defaultdict, deque

files_by_repo = {repo: set(g["path"]) for repo, g in files_df.groupby("repository_name")}

RELATIVE_IMPORT_RE = re.compile(r"^from\s+(\.+)(\S*)\s+import")
ABSOLUTE_IMPORT_RE = re.compile(r"^(?:from|import)\s+([\w\.]+)")
JS_RELATIVE_IMPORT_RE = re.compile(r"""['"](\.{1,2}/[^'"]+)['"]""")


def resolve_python_import(source_file, target_text, existing_files):
    text = target_text.strip()
    source_dir = PurePosixPath(source_file).parent
    m = RELATIVE_IMPORT_RE.match(text)
    if m:
        dots, module_path = m.groups()
        base_dir = source_dir
        for _ in range(len(dots) - 1):
            base_dir = base_dir.parent
        candidate_base = str(base_dir / module_path.replace(".", "/")) if module_path else str(base_dir)
        candidate_base = candidate_base.strip("/")
        for suffix in [".py", "/__init__.py"]:
            candidate = (candidate_base + suffix).lstrip("/")
            if candidate in existing_files:
                return candidate
        return None
    m = ABSOLUTE_IMPORT_RE.match(text)
    if m:
        candidate_base = m.group(1).replace(".", "/")
        for suffix in [".py", "/__init__.py"]:
            candidate = candidate_base + suffix
            if candidate in existing_files:
                return candidate
    return None


def resolve_js_import(source_file, target_text, existing_files):
    m = JS_RELATIVE_IMPORT_RE.search(target_text)
    if not m:
        return None
    rel_path = m.group(1)
    source_dir = PurePosixPath(source_file).parent
    candidate_base = str(source_dir / rel_path)
    for suffix in ["", ".js", ".jsx", ".ts", ".tsx", "/index.js", "/index.ts"]:
        candidate = (candidate_base + suffix).lstrip("./").lstrip("/")
        if candidate in existing_files:
            return candidate
    return None


def resolve_import(repo_name, source_file, target_text, extension):
    existing = files_by_repo.get(repo_name, set())
    return resolve_python_import(source_file, target_text, existing) if extension == ".py" \
        else resolve_js_import(source_file, target_text, existing)


imports_df = relationships_df[relationships_df["relation_type"] == "imports"].merge(
    files_df[["repository_name", "path", "extension"]],
    left_on=["repository_name", "source_file"], right_on=["repository_name", "path"], how="left"
)
imports_df["resolved_target"] = imports_df.apply(
    lambda r: resolve_import(r["repository_name"], r["source_file"], r["target"], r["extension"]), axis=1
)
resolved_imports = imports_df.dropna(subset=["resolved_target"])
print(f"Resolved {len(resolved_imports)} of {len(imports_df)} import statements "
      f"({len(resolved_imports)/len(imports_df)*100:.1f}%) to in-repo files.")

Resolved 2263 of 5221 import statements (43.3%) to in-repo files.


In [29]:
graph = defaultdict(lambda: defaultdict(set))
for row in resolved_imports.itertuples():
    graph[row.repository_name][row.source_file].add(row.resolved_target)
    graph[row.repository_name][row.resolved_target].add(row.source_file)


def graph_distance(repo, file_a, file_b, max_depth=3):
    """BFS shortest path, capped at max_depth. Returns max_depth+1 if unreachable
    within that many hops (treated as 'no meaningful graph relationship')."""
    if file_a == file_b:
        return 0
    repo_graph = graph.get(repo, {})
    if file_a not in repo_graph:
        return max_depth + 1
    visited = {file_a}
    queue = deque([(file_a, 0)])
    while queue:
        node, dist = queue.popleft()
        if dist >= max_depth:
            continue
        for neighbor in repo_graph.get(node, ()):
            if neighbor == file_b:
                return dist + 1
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, dist + 1))
    return max_depth + 1


print(f"Graph built for {len(graph)} repositories.")

Graph built for 3 repositories.


### 6.2 Historical co-change frequency (pre-cutoff only)

Reuses the same temporal split as Section 5 — how often have two files
changed together in the same commit, counted **only from history before the
cutoff**, so this feature can't leak information from the commits we'll use
as evaluation labels below.


In [30]:
cochange_counts = defaultdict(int)

train_by_commit = train_changes.groupby(["repository_name", "commit_sha"])["file_path"].apply(list)
for (repo, sha), file_list in train_by_commit.items():
    unique_files = sorted(set(file_list))
    if len(unique_files) < 2 or len(unique_files) > 20:  # cap to avoid O(n^2) blowup on huge commits
        continue
    for i in range(len(unique_files)):
        for j in range(i + 1, len(unique_files)):
            cochange_counts[(repo, unique_files[i], unique_files[j])] += 1

print(f"Built co-change counts for {len(cochange_counts)} file pairs.")


def get_cochange(repo, file_a, file_b):
    key = (repo, file_a, file_b) if file_a < file_b else (repo, file_b, file_a)
    return cochange_counts.get(key, 0)

Built co-change counts for 30595 file pairs.


### 6.3 File-level semantic similarity

Aggregates each file's symbol embeddings (from Section 4) into one mean
vector per file, so we can compare files rather than individual functions.


In [32]:
symbols_df["_emb_row"] = np.arange(len(symbols_df))

file_embeddings = {}
for (repo, file_path), group in symbols_df.groupby(["repository_name", "file_path"]):
    rows = group["_emb_row"].values
    mean_vec = embeddings[rows].mean(axis=0)
    norm = np.linalg.norm(mean_vec)
    if norm > 0:
        mean_vec = mean_vec / norm
    file_embeddings[(repo, file_path)] = mean_vec

print(f"Built file-level embeddings for {len(file_embeddings)} files.")


def semantic_similarity(repo, file_a, file_b):
    vec_a = file_embeddings.get((repo, file_a))
    vec_b = file_embeddings.get((repo, file_b))
    if vec_a is None or vec_b is None:
        return 0.0
    return float(np.dot(vec_a, vec_b))

Built file-level embeddings for 1132 files.


### 6.4 Build the labeled dataset from post-cutoff commits

For each post-cutoff commit touching 2+ files: one file is the "source"
(the change that triggers the query), the rest are the **true affected**
set (positive labels). Negative candidates are sampled from files in the
same repo that weren't touched in that commit. Features come only from
pre-cutoff/static signals — never from the eval commit's own co-change
info, which would leak the answer we're trying to predict.


In [33]:
eval_by_commit = eval_changes.groupby(["repository_name", "commit_sha"])["file_path"].apply(list)

rng = np.random.default_rng(42)
NEG_PER_POS = 3
impact_rows = []
query_id = 0

for (repo, sha), file_list in eval_by_commit.items():
    unique_files = sorted(set(file_list))
    if len(unique_files) < 2 or len(unique_files) > 15:
        continue
    all_repo_files = list(files_by_repo.get(repo, []))
    if len(all_repo_files) < 5:
        continue

    source_file = unique_files[0]
    true_affected = set(unique_files[1:])

    query_id += 1
    for target in true_affected:
        impact_rows.append({"query_id": query_id, "repository_name": repo,
                             "source_file": source_file, "candidate_file": target, "label": 1})

    n_neg = min(NEG_PER_POS * len(true_affected), len(all_repo_files))
    candidates = rng.choice(all_repo_files, size=min(n_neg * 3, len(all_repo_files)), replace=False)
    neg_added = 0
    for c in candidates:
        if c not in true_affected and c != source_file:
            impact_rows.append({"query_id": query_id, "repository_name": repo,
                                 "source_file": source_file, "candidate_file": c, "label": 0})
            neg_added += 1
        if neg_added >= n_neg:
            break

impact_dataset = pd.DataFrame(impact_rows)
print(f"Built impact dataset: {len(impact_dataset)} rows, {impact_dataset['query_id'].nunique()} queries")
print(f"Positive rate: {impact_dataset['label'].mean():.3f}")

Built impact dataset: 15660 rows, 1326 queries
Positive rate: 0.250


In [34]:
def compute_pair_features(row):
    repo, src, cand = row["repository_name"], row["source_file"], row["candidate_file"]
    return pd.Series({
        "graph_distance": graph_distance(repo, src, cand),
        "cochange_count": get_cochange(repo, src, cand),
        "semantic_similarity": semantic_similarity(repo, src, cand),
    })


feature_cols_df = impact_dataset.apply(compute_pair_features, axis=1)
impact_dataset = pd.concat([impact_dataset, feature_cols_df], axis=1)

print("Mean feature value by label (sanity check -- label=1 should show")
print("higher cochange_count and semantic_similarity than label=0):")
print(impact_dataset[["graph_distance", "cochange_count", "semantic_similarity", "label"]].groupby("label").mean())

Mean feature value by label (sanity check -- label=1 should show
higher cochange_count and semantic_similarity than label=0):
       graph_distance  cochange_count  semantic_similarity
label                                                     
0            3.936909        0.823244             0.042012
1            3.798978        9.125160             0.080323


### 6.5 Train the impact-ranking model

Split **by query**, not by row — every candidate for one query must stay
together, since the evaluation metrics below are computed per query.


In [35]:
from sklearn.ensemble import RandomForestClassifier

all_query_ids = impact_dataset["query_id"].unique()
train_qids, test_qids = train_test_split(all_query_ids, test_size=0.3, random_state=0)

impact_train_df = impact_dataset[impact_dataset["query_id"].isin(train_qids)]
impact_test_df = impact_dataset[impact_dataset["query_id"].isin(test_qids)].copy()

IMPACT_FEATURE_COLS = ["graph_distance", "cochange_count", "semantic_similarity"]
X_impact_train = impact_train_df[IMPACT_FEATURE_COLS].values
y_impact_train = impact_train_df["label"].values
X_impact_test = impact_test_df[IMPACT_FEATURE_COLS].values

impact_model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=0)
impact_model.fit(X_impact_train, y_impact_train)
impact_test_df["model_score"] = impact_model.predict_proba(X_impact_test)[:, 1]

print(f"Train: {len(X_impact_train)} rows ({impact_train_df['query_id'].nunique()} queries)")
print(f"Test: {len(X_impact_test)} rows ({impact_test_df['query_id'].nunique()} queries)")

Train: 11052 rows (928 queries)
Test: 4608 rows (398 queries)


### 6.6 Ranking evaluation: Precision@K, Recall@K, MAP

Compares the combined model against dependency-only and co-change-only
baselines — per the plan's requirement (§7.3, §8.2) that a combined feature
set must justify itself against each signal alone. This is a preview of the
full ablation study in Section 7.


In [36]:
def evaluate_ranking(df, score_col, k=5):
    precisions, recalls, avg_precisions = [], [], []
    for qid, group in df.groupby("query_id"):
        group = group.sort_values(score_col, ascending=False)
        y_true = group["label"].values
        n_true = y_true.sum()
        if n_true == 0:
            continue
        top_k = y_true[:k]
        precisions.append(top_k.sum() / min(k, len(y_true)))
        recalls.append(top_k.sum() / n_true)
        hits, ap_sum = 0, 0.0
        for i, is_relevant in enumerate(y_true, start=1):
            if is_relevant:
                hits += 1
                ap_sum += hits / i
        avg_precisions.append(ap_sum / n_true)
    return {
        "precision_at_k": float(np.mean(precisions)) if precisions else 0.0,
        "recall_at_k": float(np.mean(recalls)) if recalls else 0.0,
        "map": float(np.mean(avg_precisions)) if avg_precisions else 0.0,
        "n_queries": len(precisions),
    }


IMPACT_K = 5
impact_test_df["dep_only_score"] = -impact_test_df["graph_distance"]
impact_test_df["cochange_only_score"] = impact_test_df["cochange_count"]

combined_metrics = evaluate_ranking(impact_test_df, "model_score", k=IMPACT_K)
dep_only_metrics = evaluate_ranking(impact_test_df, "dep_only_score", k=IMPACT_K)
cochange_only_metrics = evaluate_ranking(impact_test_df, "cochange_only_score", k=IMPACT_K)

impact_results = {
    "k": IMPACT_K, "n_queries": combined_metrics["n_queries"],
    "dependency_only": dep_only_metrics,
    "cochange_only": cochange_only_metrics,
    "combined_model": combined_metrics,
}

print(f"=== Ranking evaluation (K={IMPACT_K}, {combined_metrics['n_queries']} test queries) ===")
for name, m in [("Dependency-only", dep_only_metrics), ("Co-change-only", cochange_only_metrics), ("Combined model", combined_metrics)]:
    print(f"{name:16s}: Precision@K={m['precision_at_k']:.3f}  Recall@K={m['recall_at_k']:.3f}  MAP={m['map']:.3f}")

=== Ranking evaluation (K=5, 398 test queries) ===
Dependency-only : Precision@K=0.473  Recall@K=0.934  MAP=0.987
Co-change-only  : Precision@K=0.460  Recall@K=0.920  MAP=0.954
Combined model  : Precision@K=0.448  Recall@K=0.903  MAP=0.905


In [37]:
with open(MODELS_DIR / "impact_model.pkl", "wb") as f:
    pickle.dump({"model": impact_model, "feature_cols": IMPACT_FEATURE_COLS}, f)

impact_dataset.to_csv(DATA_DIR / "impact_dataset.csv", index=False)
with open(METRICS_DIR / "impact.json", "w") as f:
    json.dump({**impact_results, "generated_at": datetime.now().isoformat()}, f, indent=4)

backup_to_drive()
print("Saved models/impact_model.pkl, data/impact_dataset.csv, metrics/impact.json")

Backed up to /content/drive/MyDrive/codeweave_dataset
Saved models/impact_model.pkl, data/impact_dataset.csv, metrics/impact.json


### 6.7 Query function

Given a file that changed, rank all other files in the repo by predicted
impact likelihood.


In [38]:
def predict_impact(repo, source_file, top_k=10):
    candidates = [f for f in files_by_repo.get(repo, []) if f != source_file]
    rows = []
    for cand in candidates:
        rows.append({
            "candidate_file": cand,
            "graph_distance": graph_distance(repo, source_file, cand),
            "cochange_count": get_cochange(repo, source_file, cand),
            "semantic_similarity": semantic_similarity(repo, source_file, cand),
        })
    cand_df = pd.DataFrame(rows)
    if cand_df.empty:
        return cand_df
    cand_df["impact_score"] = impact_model.predict_proba(cand_df[IMPACT_FEATURE_COLS].values)[:, 1]
    return cand_df.sort_values("impact_score", ascending=False).head(top_k)


# Example
example_repo = files_df["repository_name"].iloc[0]
example_file = files_df[files_df["repository_name"] == example_repo]["path"].iloc[0]
print(f"If {example_file} changes, likely affected:")
predict_impact(example_repo, example_file, top_k=5)

If .readthedocs.yaml changes, likely affected:


,candidate_file,graph_distance,cochange_count,semantic_similarity,impact_score
33,.pre-commit-config.yaml,4,2,0.0,0.703193
45,.github/workflows/tests.yaml,4,2,0.0,0.703193
32,docs/conf.py,4,1,0.0,0.657513
1,examples/tutorial/flaskr/db.py,4,0,0.0,0.411385
2,README.md,4,0,0.0,0.411385


## 7. Ablation Study

Tests each impact-prediction feature source in isolation and in combination,
per the implementation plan §7.3/§8.3 — this is the strongest single piece of
evidence that the combined feature set is actually earning its complexity,
rather than existing for "architectural decoration."

Reuses `impact_train_df` / `impact_test_df` / `evaluate_ranking()` from
Section 6 — same train/test split, same evaluation function, only the
feature columns change between runs.


In [39]:
ABLATION_CONFIGS = {
    "Git only": ["cochange_count"],
    "Graph only": ["graph_distance"],
    "Embeddings only": ["semantic_similarity"],
    "Git + Graph": ["cochange_count", "graph_distance"],
    "Graph + Embeddings": ["graph_distance", "semantic_similarity"],
    "Git + Graph + Embeddings": ["cochange_count", "graph_distance", "semantic_similarity"],
}

ABLATION_K = 5
ablation_results = {}

for config_name, feature_cols in ABLATION_CONFIGS.items():
    X_train = impact_train_df[feature_cols].values
    y_train = impact_train_df["label"].values
    X_test = impact_test_df[feature_cols].values

    model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=0)
    model.fit(X_train, y_train)

    scored = impact_test_df.copy()
    scored["_score"] = model.predict_proba(X_test)[:, 1]
    metrics = evaluate_ranking(scored, "_score", k=ABLATION_K)
    ablation_results[config_name] = metrics
    print(f"{config_name:28s}: Precision@{ABLATION_K}={metrics['precision_at_k']:.3f}  "
          f"Recall@{ABLATION_K}={metrics['recall_at_k']:.3f}  MAP={metrics['map']:.3f}")

Git only                    : Precision@5=0.457  Recall@5=0.917  MAP=0.947
Graph only                  : Precision@5=0.473  Recall@5=0.933  MAP=0.976
Embeddings only             : Precision@5=0.466  Recall@5=0.924  MAP=0.945
Git + Graph                 : Precision@5=0.453  Recall@5=0.911  MAP=0.926
Graph + Embeddings          : Precision@5=0.467  Recall@5=0.925  MAP=0.947
Git + Graph + Embeddings    : Precision@5=0.448  Recall@5=0.903  MAP=0.906


### Results table, sorted by MAP


In [40]:
ablation_df = pd.DataFrame(ablation_results).T.sort_values("map", ascending=False)
ablation_df.index.name = "feature_set"
ablation_df

,precision_at_k,recall_at_k,map,n_queries
feature_set,,,,
Graph only,0.472990,0.932725,0.976255,398.0
Graph + Embeddings,0.467462,0.925423,0.946720,398.0
Git only,0.456910,0.917127,0.946574,398.0
Embeddings only,0.466457,0.924450,0.944686,398.0
Git + Graph,0.453392,0.910615,0.926095,398.0
Git + Graph + Embeddings,0.447864,0.902554,0.906444,398.0


In [41]:
with open(METRICS_DIR / "ablation.json", "w") as f:
    json.dump({
        "k": ABLATION_K,
        "results": ablation_results,
        "generated_at": datetime.now().isoformat(),
    }, f, indent=4)

ablation_df.to_csv(DATA_DIR / "ablation_results.csv")
backup_to_drive()
print("Saved metrics/ablation.json and data/ablation_results.csv")

Backed up to /content/drive/MyDrive/codeweave_dataset
Saved metrics/ablation.json and data/ablation_results.csv


### How to read this table

- If **"Git + Graph + Embeddings"** isn't the top row, that's a real,
  reportable finding — it means one or more feature sources aren't adding
  value for impact prediction on this dataset (or are adding noise). Report
  it honestly rather than only showing the combination that looks best.
- If a single-feature config (e.g. "Git only") outperforms combinations,
  investigate before concluding the extra features are useless — check
  whether `semantic_similarity` is carrying real signal (it won't, if
  Section 4 hasn't been re-run with a live CodeBERT model in this session).
- This ablation only covers the **impact prediction** feature set. The plan's
  similarity benchmark (Section 4, CodeBERT vs. TF-IDF) and risk benchmark
  (Section 5, baseline vs. XGBoost) are separate ablation-style comparisons
  already done in their own sections.


## 8. RAG + Chat

Ties everything together: a plain-English question gets routed to the
right specialized model (risk / impact / similarity), which returns
structured **evidence** — and only that evidence gets turned into an answer.
Per the plan's grounding rule (§12): if there's no evidence, the chatbot
says so rather than inventing a plausible-sounding answer.

This uses a **template-based answer by default** (no API key or internet
required) so it works standalone. An optional real-LLM upgrade is at the end.


### 8.1 Intent router

Deliberately simple and keyword-based rather than LLM-based — per the plan,
intent routing is meant to be a small, testable, deterministic set of
choices (risk / impact / similarity / run), not an open-ended decision.


In [42]:
INTENT_PATTERNS = {
    "risk": re.compile(r"\brisk|risky|buggy|bug[- ]prone|dangerous\b", re.IGNORECASE),
    "impact": re.compile(r"\baffect|impact|break|depend|touch\b", re.IGNORECASE),
    "similarity": re.compile(r"\bsimilar|like this|duplicate|equivalent\b", re.IGNORECASE),
    "run": re.compile(r"\brun|reproduce|set ?up|install\b", re.IGNORECASE),
}


def detect_intent(question):
    for intent, pattern in INTENT_PATTERNS.items():
        if pattern.search(question):
            return intent
    return "unknown"

### 8.2 Entity extraction

Finds which known file or symbol the question is actually about, by
matching against `files_df`/`symbols_df` — no guessing, no fuzzy inference.
If nothing matches, the chatbot will abstain rather than pick something
arbitrary (tested below).


In [43]:
def extract_file_entity(question, repo_name=None):
    candidates = files_df if repo_name is None else files_df[files_df["repository_name"] == repo_name]
    for path in candidates["path"]:
        if path.split("/")[-1] in question:
            row = candidates[candidates["path"] == path].iloc[0]
            return row["repository_name"], path
    return None, None


def extract_symbol_entity(question, repo_name=None):
    candidates = symbols_df if repo_name is None else symbols_df[symbols_df["repository_name"] == repo_name]
    ordered = candidates.reindex(candidates["name"].str.len().sort_values(ascending=False).index)
    for _, row in ordered.iterrows():
        if re.search(r"\b" + re.escape(row["name"]) + r"\b", question):
            return row["repository_name"], row["symbol_id"], row["name"]
    return None, None, None

### 8.3 Tool wrappers

Each wrapper calls an already-trained model/index and returns a structured
**evidence object** — matching the plan's evidence contract (§12.2): source
type, repository, file/symbol identifiers, the score, and (where available)
reason codes. Nothing here is prose yet — that's the job of `format_answer()`
below, which is only allowed to describe what's in the evidence object.


In [44]:
def tool_get_risk(repo_name, file_path):
    row = train_features[(train_features["repository_name"] == repo_name) & (train_features["path"] == file_path)]
    if row.empty:
        return None
    row = row.iloc[0]
    X = row[FEATURE_COLS].values.reshape(1, -1).astype(float)
    score = float(xgb_model.predict_proba(X)[0, 1])
    shap_values = explainer.shap_values(X)[0]
    contributions = sorted(zip(FEATURE_COLS, shap_values), key=lambda x: abs(x[1]), reverse=True)
    return {
        "source_type": "risk_model", "repository": repo_name, "file": file_path,
        "risk_score": score,
        "top_factors": [{"feature": f, "direction": "increases" if v > 0 else "decreases"} for f, v in contributions[:3]],
    }


def tool_get_impact(repo_name, file_path, top_k=5):
    if repo_name not in files_by_repo or file_path not in files_by_repo[repo_name]:
        return None
    top = predict_impact(repo_name, file_path, top_k=top_k)
    if top.empty:
        return None
    return {
        "source_type": "impact_model", "repository": repo_name, "source_file": file_path,
        "affected": top[["candidate_file", "impact_score"]].to_dict(orient="records"),
    }


def tool_get_similarity(symbol_id, symbol_name, top_k=5):
    results = find_similar(symbol_id, k=top_k)
    if not results:
        return None
    return {
        "source_type": "similarity_search", "query_symbol": symbol_name,
        "similar": results,
    }

### 8.4 Answer formatting (evidence -> prose, nothing invented)


In [45]:
def format_answer(intent, evidence):
    if evidence is None:
        return "I don't have evidence for that in the current analysis."

    if intent == "risk":
        score = evidence["risk_score"]
        level = "high" if score >= 0.66 else "medium" if score >= 0.33 else "low"
        factors = ", ".join(f["feature"] for f in evidence["top_factors"])
        return (f"`{evidence['file']}` has {level} predicted risk (score={score:.2f}), "
                f"driven mainly by: {factors}.")

    if intent == "impact":
        affected = ", ".join(f"{a['candidate_file']} ({a['impact_score']:.2f})" for a in evidence["affected"][:3])
        return f"If `{evidence['source_file']}` changes, likely affected: {affected}."

    if intent == "similarity":
        matches = ", ".join(f"{s['name']} ({s['repository_name']}/{s['file_path']}, sim={s['similarity']:.2f})" for s in evidence["similar"][:3])
        return f"Symbols similar to `{evidence['query_symbol']}`: {matches}."

    return "I don't have a way to answer that yet."

### 8.5 Orchestrator


In [46]:
def ask_synode(question, default_repo=None):
    intent = detect_intent(question)

    if intent == "unknown":
        return {"answer": "I can help with questions about risk, change impact, or code similarity "
                           "for a specific file or function. Could you name one and ask about it?",
                "evidence": None, "intent": intent}

    if intent == "run":
        return {"answer": "The repository setup runner isn't built yet in this notebook series — "
                           "that's a separate module (Repository Setup Runner) not yet implemented here.",
                "evidence": None, "intent": intent}

    if intent == "similarity":
        repo, symbol_id, symbol_name = extract_symbol_entity(question, default_repo)
        if symbol_id is None:
            return {"answer": "I couldn't identify which function/class you mean. "
                               "Try naming it directly.", "evidence": None, "intent": intent}
        evidence = tool_get_similarity(symbol_id, symbol_name)
        return {"answer": format_answer(intent, evidence), "evidence": evidence, "intent": intent}

    # risk / impact both need a file entity
    repo, file_path = extract_file_entity(question, default_repo)
    if repo is None and default_repo:
        repo = default_repo
    if file_path is None:
        return {"answer": f"I understood you're asking about {intent}, but couldn't identify which file. "
                           "Try including the filename.", "evidence": None, "intent": intent}

    if intent == "risk":
        evidence = tool_get_risk(repo, file_path)
    else:
        evidence = tool_get_impact(repo, file_path)

    return {"answer": format_answer(intent, evidence), "evidence": evidence, "intent": intent}

### 8.6 Test it

Includes the two cases that matter most for the grounding rule: an
off-topic question (should get no evidence) and a question with no
identifiable file (should abstain, not guess).


In [48]:
example_repo = files_df["repository_name"].iloc[0]
example_file = files_df[files_df["repository_name"] == example_repo]["path"].iloc[20]
example_filename = example_file.split("/")[-1]

test_questions = [
    f"Why is {example_filename} risky?",
    f"What breaks if I change {example_filename}?",
    "What's the capital of France?",   # off-topic -> no evidence
    "Why is this risky?",              # no file named -> abstain
]

for q in test_questions:
    result = ask_synode(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"   [intent={result['intent']}, has_evidence={result['evidence'] is not None}]\n")

Q: Why is test_cli.py risky?
A: `tests/test_cli.py` has high predicted risk (score=0.91), driven mainly by: loc, days_since_last_change, complexity.
   [intent=risk, has_evidence=True]

Q: What breaks if I change test_cli.py?
A: If `tests/test_cli.py` changes, likely affected: tests/test_async.py (0.97), tests/test_request.py (0.94), tests/conftest.py (0.92).
   [intent=impact, has_evidence=True]

Q: What's the capital of France?
A: I can help with questions about risk, change impact, or code similarity for a specific file or function. Could you name one and ask about it?
   [intent=unknown, has_evidence=False]

Q: Why is this risky?
A: I understood you're asking about risk, but couldn't identify which file. Try including the filename.
   [intent=risk, has_evidence=False]



### 8.7 Optional: real LLM for nicer phrasing

Everything above already works standalone with no API key. This section
only *rephrases* the same evidence more naturally — the LLM is explicitly
instructed to use nothing but the evidence object, matching the plan's
grounding rule that the LLM explains, it doesn't invent. Skip this cell
entirely if you don't want to set up an API key.


In [49]:
USE_LLM = False  # flip to True after adding an API key below

if USE_LLM:
    !pip -q install anthropic
    import anthropic
    from getpass import getpass

    api_key = getpass("Anthropic API key: ")
    llm_client = anthropic.Anthropic(api_key=api_key)

    def llm_explain(question, intent, evidence):
        if evidence is None:
            return None  # nothing to explain -- fall back to the template answer
        prompt = (
            "You are explaining a software analysis result to a developer. "
            "Use ONLY the evidence JSON below -- do not invent any fact not present in it. "
            "Keep it to 2-3 sentences.\n\n"
            f"Question: {question}\n"
            f"Intent: {intent}\n"
            f"Evidence: {json.dumps(evidence)}"
        )
        response = llm_client.messages.create(
            model="claude-sonnet-4-5", max_tokens=300,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

    def ask_synode_llm(question, default_repo=None):
        result = ask_synode(question, default_repo)
        llm_answer = llm_explain(question, result["intent"], result["evidence"])
        if llm_answer is not None:
            result["answer"] = llm_answer
        return result

    print("LLM-backed chat ready: use ask_synode_llm(question) instead of ask_synode(question)")
else:
    print("USE_LLM is False -- using template-based answers only (ask_synode). "
          "Set USE_LLM = True and re-run this cell to enable real LLM phrasing.")

USE_LLM is False -- using template-based answers only (ask_synode). Set USE_LLM = True and re-run this cell to enable real LLM phrasing.


In [50]:
backup_to_drive()
print("Sections 7 and 8 complete.")

Backed up to /content/drive/MyDrive/codeweave_dataset
Sections 7 and 8 complete.
